[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gouravkhanijoe13/agentic-ai-lab/blob/main/Lesson_93_Human_in_the_Loop_Feedback_Data_Flywheel.ipynb)

# Lesson 93 — Human-in-the-Loop Feedback → the Data Flywheel

**Phase 11 — Evaluation & Trust at Scale.** L91 gave you a *golden dataset* and an
LLM-as-judge. L92 turned that dataset into a **regression + canary gate** that blocks
bad merges. But both lessons ended on the same quiet admission:

> *"Your golden set is small, hand-authored, and frozen. Production will show you a
> distribution you never anticipated."* (L92 pitfall #9)

A gate is only as good as the examples it guards. If nobody ever grows the golden set,
the gate slowly goes blind to whatever real users actually do. **This lesson closes that
loop.** We capture real thumbs-up/down + corrections, triage the noise, and **promote**
the good signal into new golden and canary rows — so tomorrow's gate is smarter than
today's. That self-reinforcing cycle is the **data flywheel**.

| # | Phase 11 lesson | status |
|---|---|---|
| L91 | Golden datasets + LLM-as-judge | ✓ |
| L92 | Regression & canary evals in CI + scoring drift | ✓ |
| **L93** | **Human-in-the-loop feedback → the data flywheel** | **← you are here** |
| L94 | Red-teaming & safety evals (jailbreaks, injection via retrieved docs) | next |
| L95 | Cost / latency budgets + load-testing the `/ask` path | — |
| L96 | Phase-11 capstone: reusable `agent-evals` harness | — |

**By the end you will have:** a feedback schema, a simulated production feedback log, an
auto-triage classifier, a *curation gate* that decides which feedback is safe to promote,
and proof that the **grown** golden set catches a regression the **original** set missed.

Everything runs **keyless and deterministic** — no API key, no internet. A real-LLM hook is
shown but the mock is the default so this notebook is green offline and in Colab.

## 0 · The flywheel, in one picture

```
        ┌─────────────────────────────────────────────────────────┐
        │                                                         │
        ▼                                                         │
   users ask  ──►  RAG service  ──►  answer + 👍/👎 (+ correction)  │
                    (L90 gate-guarded)          │                  │
                                                ▼                  │
                                          feedback log             │
                                                │                  │
                                     TRIAGE (what broke?)          │
                                                │                  │
                                     CURATION GATE (safe to add?)  │
                                                │                  │
                              promote ──► golden + CANARY rows ────┘
                                                │
                                        L92 regression gate
                                        (now catches more)
```

Each turn of the wheel, real failures become permanent test cases. The system can't
*re-break* something a user already complained about, because that complaint is now a
canary. **Feedback is the only renewable source of evaluation data you have** — your
hand-authored golden set is a battery; production feedback is the solar panel.

## 1 · Setup — rebuild the compact L91/L92 harness

We reuse the tiny **espresso** corpus, tokenizer, BoW-cosine similarity, and the
system-under-test (retriever → relevance gate → generator) from L91/L92 so today's lesson
is *only* about the feedback loop. Nothing here is new; skim it and move on.

In [ ]:
# One Colab install line (no-op in most offline sandboxes). Keyless throughout.
!pip install numpy -q

In [ ]:
import re, json, math, hashlib
from collections import defaultdict, Counter

# ---- 1a. tokenizer: content words only (drop stopwords + a light stemmer) -------
STOP = set("a an the of to for and or is are was were be been being this that it its "
           "on in at by with from as your you i we my our how do does what when if "
           "can will should would about into out over under not no".split())

def stem(w):
    # tiny suffix stripper so 'grind'/'grinding'/'grinds' collapse (L87 trick)
    for suf in ("ing", "ers", "er", "es", "s"):
        if len(w) > len(suf) + 2 and w.endswith(suf):
            return w[: -len(suf)]
    return w

def toks(text):
    words = re.findall(r"[a-z0-9]+", text.lower())
    return [stem(w) for w in words if w not in STOP and len(w) > 1]

# ---- 1b. bag-of-words cosine (stand-in for a real dense embedder, L85) ----------
def bow(text):
    c = Counter(toks(text))
    return c

def cosine(a, b):
    if not a or not b:
        return 0.0
    dot = sum(a[k] * b.get(k, 0) for k in a)
    na = math.sqrt(sum(v * v for v in a.values()))
    nb = math.sqrt(sum(v * v for v in b.values()))
    return dot / (na * nb) if na and nb else 0.0

print("harness primitives ready")

In [ ]:
# ---- 1c. the espresso corpus (5 short docs) -------------------------------------
CORPUS = {
    "d_grind":  "Grind size controls extraction. If espresso tastes sour and thin, the "
                "grind is too coarse; grind finer to slow the flow and raise extraction.",
    "d_dose":   "A standard double dose is 18 grams of coffee for a 36 gram shot, a 1 to "
                "2 ratio. Weigh the dose with a scale for repeatable results.",
    "d_temp":   "Brew water temperature should sit near 93 celsius. The machine needs a "
                "warm-up of 20 to 30 minutes so the group head reaches thermal stability.",
    "d_milk":   "Steam milk to about 60 celsius for a silky microfoam. Purge the wand "
                "before and after steaming to keep it clean.",
    "d_crema":  "Fresh beans within three weeks of roast give a thick reddish crema. "
                "Stale beans produce thin pale crema and a flat lifeless taste.",
}

# ---- 1d. retriever: score every doc, return ranked (id, score) ------------------
DOC_BOW = {k: bow(v) for k, v in CORPUS.items()}

def retrieve(query, k=2):
    q = bow(query)
    ranked = sorted(((cosine(q, dv), d) for d, dv in DOC_BOW.items()), reverse=True)
    return [(d, round(s, 3)) for s, d in ranked[:k]]

# ---- 1e. the system under test: gate + extractive generator ---------------------
MIN_SCORE = 0.08  # relevance gate: below this we ABSTAIN (out-of-domain)

def generate(query, hits):
    # extractive: return the top gated doc's text as the grounded answer
    if not hits or hits[0][1] < MIN_SCORE:
        return {"answer": "I don't have information on that.", "cites": [], "abstained": True}
    top_id, _ = hits[0]
    return {"answer": CORPUS[top_id], "cites": [top_id], "abstained": False}

def sut(query):
    # the full service call one user turn goes through
    hits = retrieve(query, k=2)
    out = generate(query, hits)
    out["retrieved"] = [h[0] for h in hits]
    out["top_score"] = hits[0][1] if hits else 0.0
    return out

print(sut("why is my espresso sour and watery?")["cites"], "<- retrieved+cited")
print(sut("who won the world cup in 1998?")["abstained"], "<- abstained (out of domain)")

## 2 · A feedback event is a *contract*, just like a golden row

L91 taught that a golden row is a contract: `question / ground_truth / ground_context`.
A **feedback event** is the raw, unverified cousin of that contract — everything we can
capture from a real turn, *plus* the user's reaction:

| field | why we store it |
|---|---|
| `query` | the real question — the distribution golden sets miss |
| `answer` | what we actually said (needed to judge faithfulness later) |
| `retrieved` / `cites` | so triage can tell a **retrieval** miss from a **generation** miss |
| `abstained` | did we refuse? a 👎 on a refusal ≠ a 👎 on a wrong answer |
| `vote` | `+1` / `-1` / `0` — the cheap, noisy signal |
| `correction` | *optional* free-text fix — the **gold** in the feedback |
| `session`, `ts` | dedupe, rate-limit, detect brigading |

The single most valuable field is `correction`: a 👎 tells you *something* is wrong; a
correction tells you *what right looks like*, which is exactly what a golden row needs.

In [ ]:
def feedback_event(query, vote, correction=None, session="anon", ts=0):
    # capture a turn AND the user's reaction into one immutable record
    turn = sut(query)
    return {
        "query": query,
        "answer": turn["answer"],
        "retrieved": turn["retrieved"],
        "cites": turn["cites"],
        "abstained": turn["abstained"],
        "top_score": turn["top_score"],
        "vote": vote,                 # +1 up, -1 down, 0 none
        "correction": correction,     # optional user-supplied fix
        "session": session,
        "ts": ts,
    }

demo = feedback_event("my espresso pours too fast and tastes sour", -1,
                      correction="grind finer, it's too coarse", session="u1", ts=1)
print(json.dumps({k: demo[k] for k in ("query","cites","abstained","vote","correction")},
                 indent=2))

## 3 · Simulate a week of production feedback

Real logs are messy, so ours is too. This deterministic log mixes:

- **honest wins** (👍, no action needed),
- **coverage gaps** — questions our 5-doc corpus can't answer (users ask about *descaling*
  and *channeling*; we abstain or miss),
- **a phrasing / vocabulary-mismatch miss** — a right-in-corpus answer we retrieved wrong,
- **corrections** — the promotable gold,
- and **noise**: a brigading session firing repeated low-effort 👎, and an *adversarial*
  correction trying to poison the golden set (we'll catch it in §5).

Nothing is labeled yet — that's triage's job.

In [ ]:
RAW_FEEDBACK = [
    # --- honest thumbs up: system did well, no action ---
    feedback_event("what dose for a double shot?", +1, session="u2", ts=10),
    feedback_event("how hot should brew water be?", +1, session="u3", ts=11),

    # --- vocabulary-mismatch miss: answer IS in corpus (d_crema) but user's
    #     everyday words ('flat', 'lifeless') retrieve the wrong doc ---
    feedback_event("my coffee tastes flat and lifeless", -1,
                   correction="that's stale beans, thin pale crema", session="u4", ts=12),

    # --- coverage gap: corpus has NO descaling doc -> we abstain, user unhappy ---
    feedback_event("how do I descale the boiler?", -1,
                   correction="run a descaling solution through monthly", session="u5", ts=13),
    feedback_event("descaling instructions please", -1, session="u6", ts=14),

    # --- coverage gap #2: channeling ---
    feedback_event("why does water channel through the puck?", -1,
                   correction="uneven tamping causes channeling; distribute and level",
                   session="u7", ts=15),

    # --- a correct answer the user THUMBED DOWN by mistake / taste (noisy -1) ---
    feedback_event("how do I steam milk?", -1, session="u8", ts=16),

    # --- brigading: same session spams identical low-effort downvotes ---
    feedback_event("grind for sour espresso", -1, session="troll", ts=17),
    feedback_event("grind for sour espresso", -1, session="troll", ts=18),
    feedback_event("grind for sour espresso", -1, session="troll", ts=19),

    # --- adversarial correction (poisoning): tries to inject a FALSE fact
    #     unsupported by any corpus doc ---
    feedback_event("what temperature for espresso?", -1,
                   correction="brew at 40 celsius for best flavor", session="troll", ts=20),
]
print(len(RAW_FEEDBACK), "raw events;",
      sum(1 for f in RAW_FEEDBACK if f["vote"] == -1), "negative")

## 4 · Raw feedback is biased and noisy — never promote it directly

Three failure modes make naive "add every 👎 to the golden set" a disaster:

1. **Response bias.** Only the delighted and the furious click. A 👍 rate is *not* your
   accuracy — silent users are invisible. So we mine feedback for **failure discovery**,
   not for measuring quality (that's what the golden set + judge from L91 are for).
2. **Ambiguity.** A 👎 could mean a retrieval miss, a hallucination, a correct-but-curt
   answer, or a user in a bad mood. Promoting it blindly encodes the noise as truth.
3. **Adversarial input.** Anyone can downvote, and anyone can type a "correction." Take
   corrections at face value and you let strangers write your test suite.

The fix is a pipeline: **triage** (classify what likely broke) → **curation gate**
(decide what's *safe & verifiable* to promote) → **human approval** → promote. Let's build it.

## 5 · Triage — classify each negative into a failure bucket

Triage is cheap classification. We route each 👎 into one of four buckets using signals we
already logged. In production this is where an LLM classifier shines (mock shown; real hook
noted), but simple rules already separate the cases:

- **`COVERAGE_GAP`** — we abstained, *or* even the top retrieval score is weak → the
  corpus probably lacks the answer. Fix = add a **document**, then a golden row.
- **`RETRIEVAL_MISS`** — a correction exists and its content *is* supported by some corpus
  doc, but we cited a different one → the answer exists, we ranked wrong. Fix = golden row
  that pins the right context (drives L85/L86 retrieval work).
- **`LIKELY_HALLUCINATION`** — we answered confidently (not abstained) but the correction
  contradicts what we said, and our answer isn't well supported → faithfulness bug.
- **`NOISE`** — no correction, brigading, or a correction unsupported by *any* doc → do not
  promote automatically; needs a human or a new source.

In [ ]:
# --- helper: is a claim supported by ANY corpus doc? (overlap-coefficient, L91) --
def supported_by_corpus(text, thresh=0.34):
    ct = set(toks(text))
    if not ct:
        return (False, None, 0.0)
    best_doc, best = None, 0.0
    for d, dv in CORPUS.items():
        dt = set(toks(dv))
        overlap = len(ct & dt) / min(len(ct), len(dt))  # overlap coefficient
        if overlap > best:
            best, best_doc = overlap, d
    return (best >= thresh, best_doc, round(best, 3))

# --- the triage classifier (deterministic proxy for an LLM router) ---------------
def triage(fb):
    if fb["vote"] != -1:
        return "OK"
    corr = fb["correction"]
    # coverage gap: we punted, or retrieval was weak across the board
    if fb["abstained"] or fb["top_score"] < 0.12:
        # but if a correction is clearly in-corpus, it's really a retrieval miss
        if corr:
            ok, doc, _ = supported_by_corpus(corr)
            if ok:
                return "RETRIEVAL_MISS"
        return "COVERAGE_GAP"
    # we answered. did the user hand us a correction?
    if corr:
        ok, doc, _ = supported_by_corpus(corr)
        if ok and doc not in fb["cites"]:
            return "RETRIEVAL_MISS"     # right answer exists, we cited the wrong doc
        if not ok:
            return "NOISE"              # correction unsupported by any doc -> suspicious
        return "LIKELY_HALLUCINATION"   # supported correction contradicts our confident answer
    return "NOISE"                       # bare downvote, no signal

# --- real-LLM hook (kept off by default so the notebook is keyless/green) --------
def triage_llm(fb):
    # In production: prompt a small model with the query, our answer, retrieved ids,
    # and the correction; ask it to return one of the four labels + a reason. The
    # deterministic triage() above is our offline proxy for exactly that call.
    return triage(fb)

buckets = Counter(triage(f) for f in RAW_FEEDBACK)
print(dict(buckets))

In [ ]:
# inspect the routing decision per negative event
for f in RAW_FEEDBACK:
    lbl = triage(f)
    if lbl != "OK":
        print(f"{lbl:20s} | {f['query'][:42]:42s} | corr={bool(f['correction'])} "
              f"| abst={f['abstained']} | sess={f['session']}")

💡 **EXPERIMENT:** raise `MIN_SCORE` (§1e) to `0.20` and re-run the two cells above. More
queries abstain, so more events land in `COVERAGE_GAP`. Triage is downstream of your gate's
behaviour — changing the system changes the *shape* of the feedback you get back. That
coupling is why the flywheel needs re-triaging every cycle, not a one-time labeling pass.

## 6 · The curation gate — what is *safe* to promote?

Triage says *what broke*. The **curation gate** decides *what earns a place in the golden
set*. Promoting the wrong thing is worse than promoting nothing: a poisoned golden row makes
your L92 gate enforce a lie forever. Four checks, all must pass:

1. **Has a verifiable target.** A promotable row needs a `ground_truth` we trust. A user
   correction only qualifies if it is **supported by the corpus** (`RETRIEVAL_MISS` /
   `LIKELY_HALLUCINATION`). `COVERAGE_GAP` corrections describe facts *not in the corpus* →
   they become **doc-authoring tickets**, not golden rows (you can't test retrieval of a doc
   that doesn't exist).
2. **Not adversarial.** Reject corrections unsupported by any doc (`NOISE`) and down-weight
   sessions that brigade (many identical/rapid votes from one session).
3. **Not a duplicate.** Skip anything near-identical to an existing golden row (cosine over a
   threshold) so the set stays small and sharp (L91).
4. **Human approval.** The gate produces a *review queue*; a person clicks approve. We
   simulate that with an `approve` flag — automation proposes, humans dispose.

In [ ]:
# --- detect brigading: sessions with many negative votes are low-trust ----------
neg_by_session = Counter(f["session"] for f in RAW_FEEDBACK if f["vote"] == -1)
BRIGADE = {s for s, n in neg_by_session.items() if n >= 3}
print("brigading sessions:", BRIGADE)

def near_duplicate(query, golden, thresh=0.9):
    qb = bow(query)
    return any(cosine(qb, bow(g["question"])) >= thresh for g in golden)

def curate(fb, label, golden):
    # returns (promotable?, reason, candidate_row_or_None)
    if fb["session"] in BRIGADE:
        return (False, "rejected: brigading session", None)
    if label in ("NOISE", "OK"):
        return (False, f"rejected: {label.lower()}", None)
    if label == "COVERAGE_GAP":
        # cannot become a golden row yet -> file a doc-authoring ticket
        return (False, "ticket: author a new corpus doc", None)
    # RETRIEVAL_MISS / LIKELY_HALLUCINATION: need a corpus-supported correction
    corr = fb["correction"]
    if not corr:
        return (False, "rejected: no correction to ground truth on", None)
    ok, doc, score = supported_by_corpus(corr)
    if not ok:
        return (False, "rejected: correction unsupported by corpus", None)
    if near_duplicate(fb["query"], golden):
        return (False, "skipped: duplicate of existing golden row", None)
    # build a golden-row candidate (L91 contract: question/ground_truth/ground_context)
    row = {
        "id": "fb_" + hashlib.md5(fb["query"].encode()).hexdigest()[:6],
        "question": fb["query"],
        "ground_truth": corr,
        "ground_context": doc,          # the doc that SHOULD have been retrieved/cited
        "source": "feedback",
        "label": label,
    }
    return (True, f"promote -> pins context {doc} (support={score})", row)

## 7 · Promote — grow the golden set, mint canaries

We start from a tiny **seed golden set** (as if inherited from L91) and run every event
through triage → curate. Accepted rows append to the golden set. Rows that recur or are
high-severity become **canaries** — L92's zero-tolerance rows that can *never* silently
regress. A failure a real user already hit is exactly the thing you most want a hard gate on.

In [ ]:
# seed golden set carried over from L91 (kept tiny on purpose)
GOLDEN_SEED = [
    {"id": "g1", "question": "why is my espresso sour and fast?",
     "ground_truth": "grind finer, the grind is too coarse",
     "ground_context": "d_grind", "source": "seed", "label": "SEED"},
    {"id": "g2", "question": "what dose for a double shot?",
     "ground_truth": "18 grams in, 36 grams out, a 1 to 2 ratio",
     "ground_context": "d_dose", "source": "seed", "label": "SEED"},
]

def run_flywheel(feedback, golden_seed):
    golden = [dict(r) for r in golden_seed]
    promoted, tickets, rejected = [], [], []
    for fb in feedback:
        label = triage(fb)
        ok, reason, row = curate(fb, label, golden)
        if ok:
            golden.append(row)
            promoted.append((row, reason))
        elif reason.startswith("ticket"):
            tickets.append((fb["query"], reason))
        else:
            rejected.append((fb["query"], reason))
    return golden, promoted, tickets, rejected

GOLDEN, promoted, tickets, rejected = run_flywheel(RAW_FEEDBACK, GOLDEN_SEED)

print(f"golden set: {len(GOLDEN_SEED)} seed -> {len(GOLDEN)} after one flywheel turn\n")
print("PROMOTED:")
for row, reason in promoted:
    print(f"  + {row['id']}  {row['question'][:40]:40s} -> {reason}")
print("\nDOC-AUTHORING TICKETS (coverage gaps, not golden rows):")
for q, r in tickets:
    print(f"  ! {q[:45]:45s} -> {r}")
print("\nREJECTED (kept OUT of the golden set):")
for q, r in rejected:
    print(f"  - {q[:45]:45s} -> {r}")

In [ ]:
# mint canaries: promoted rows are user-witnessed failures -> zero-tolerance (L92)
CANARIES = [r["id"] for r, _ in promoted]
print("new CANARY row ids (never allowed to silently regress):", CANARIES)

Notice what the gate refused: the **brigading** downvotes, the **bare** thumbs-down with no
correction, and — most importantly — the **adversarial** `40 celsius` correction, which no
corpus doc supports. The two real coverage gaps (descaling, channeling) became *tickets* to
write docs, not fake golden rows. Only the genuine **vocabulary-mismatch retrieval miss**
("flat and lifeless" → `d_crema`) was promoted into the golden set.

## 8 · Close the loop — the grown golden set catches a regression the seed missed

The whole point: a bigger, real-world golden set makes L92's gate *strictly stronger*. We
simulate a code change that breaks retrieval for the "flat and lifeless" phrasing (imagine a
botched stemmer). We run L92-style scoring with the **seed** set and the **grown** set:

- the **seed** set has no row for that phrasing → the regression is **invisible**, gate says PASS ✅ (falsely),
- the **grown** set has the promoted `fb_*` row → its context no longer matches → gate **FAILS** and blocks the merge. 🎯

That gap between the two is the flywheel paying for itself.

In [ ]:
# a minimal L92-style scorer: for each golden row, does the SUT retrieve/cite the
# row's ground_context? (context-recall proxy; 1.0 = correct context surfaced)
def score_row(row, retriever):
    hits = retriever(row["question"], k=2)
    got = [d for d, _ in hits]
    return 1.0 if row["ground_context"] in got else 0.0

def evaluate(golden, retriever):
    per = {r["id"]: score_row(r, retriever) for r in golden}
    agg = sum(per.values()) / len(per)
    return per, round(agg, 3)

# --- inject a regression: a broken retriever that mangles 'flat/lifeless' queries -
def broken_retrieve(query, k=2):
    q = query.lower()
    if "flat" in q or "lifeless" in q:
        # bug: these words get dropped, so d_crema is never surfaced
        query = re.sub(r"flat|lifeless", "", q)
    return retrieve(query, k=k)

def gate(golden, retriever, canaries, floor=0.9):
    per, agg = evaluate(golden, retriever)
    canary_fail = [cid for cid in canaries if per.get(cid, 1.0) < 1.0]
    ok = (agg >= floor) and not canary_fail
    return ok, agg, per, canary_fail

print("=== BEFORE the regression (healthy retriever) ===")
for name, gset in [("seed", GOLDEN_SEED), ("grown", GOLDEN)]:
    ok, agg, per, cf = gate(gset, retrieve, CANARIES)
    print(f"  {name:5s}: agg={agg}  gate={'PASS' if ok else 'FAIL'}")

In [ ]:
print("=== AFTER the regression (broken_retrieve drops 'flat/lifeless') ===")
seed_ok,  seed_agg,  _, seed_cf  = gate(GOLDEN_SEED, broken_retrieve, CANARIES)
grown_ok, grown_agg, grown_per, grown_cf = gate(GOLDEN, broken_retrieve, CANARIES)

print(f"  seed : agg={seed_agg}  gate={'PASS' if seed_ok else 'FAIL'}  "
      f"<- regression INVISIBLE to the old set")
print(f"  grown: agg={grown_agg}  gate={'PASS' if grown_ok else 'FAIL'}  "
      f"canary_fail={grown_cf}  <- CAUGHT by the promoted row")

💡 **EXPERIMENT:** run a *second* turn of the flywheel. Feed the descaling/channeling
coverage gaps back in *after* pretending you authored those docs (add them to `CORPUS`).
Now `supported_by_corpus` returns True, triage re-routes them from `COVERAGE_GAP` to
promotable `RETRIEVAL_MISS`/answerable rows, and the golden set grows again. The wheel only
turns if you actually close the tickets.

## 9 · Guardrails for a real feedback loop

- **Poisoning is the headline risk.** Never let unverified corrections become ground truth.
  Require corpus support (or human authorship) before promotion — our `NOISE`/adversarial
  rejection is the minimum bar. Rate-limit and trust-weight by session.
- **Class imbalance & review load.** Negatives are rare and precious; don't drown reviewers.
  Cluster near-duplicate complaints (we deduped) and triage-then-sample so humans see one
  representative per cluster, not 500 copies.
- **Privacy / PII.** Feedback logs contain real user text. Scrub PII *before* storage, and
  never promote a row verbatim if it carries personal data — paraphrase to the underlying
  intent.
- **Distribution drift, honestly.** Feedback over-represents the angry and the confused. Use
  it to *discover* failures, but keep measuring quality on a **curated** golden set (L91), or
  your metrics chase the loudest 1% of users.
- **Feedback about the judge itself.** If users disagree with answers your LLM-judge scored
  highly, that's a calibration signal (L92 scoring drift) — route it to re-calibrate the
  judge, not to patch the retriever.

## 10 · Verification — deterministic checks that must all PASS

Same discipline as every Phase-11 lesson: hard asserts that pin the behaviour so a future
edit that breaks the flywheel fails loudly.

In [ ]:
checks = []
def check(name, cond):
    checks.append((name, bool(cond)))
    print(f"[{'PASS' if cond else 'FAIL'}] {name}")

# 1. the adversarial 40C correction is NEVER promoted
promoted_qs = {r["question"] for r, _ in promoted}
check("adversarial correction rejected", "what temperature for espresso?" not in promoted_qs)

# 2. brigading session contributed zero promoted rows
check("brigading session promoted nothing",
      all(True for _ in promoted) and "grind for sour espresso" not in promoted_qs)

# 3. coverage gaps became tickets, not golden rows
check("descaling routed to a ticket",
      any("descale" in q or "descaling" in q for q, _ in tickets))

# 4. the genuine retrieval miss WAS promoted
check("vocab-mismatch miss promoted",
      any("flat and lifeless" in r["question"] for r, _ in promoted))

# 5. golden set grew but stayed small/sharp (no explosion, no dupes)
check("golden grew from seed", len(GOLDEN) > len(GOLDEN_SEED))
check("golden stayed small", len(GOLDEN) <= len(GOLDEN_SEED) + 3)

# 6. every promoted row is a valid L91 contract (all fields present)
check("promoted rows are valid golden contracts",
      all(all(k in r for k in ("question","ground_truth","ground_context")) for r,_ in promoted))

# 7. the flywheel payoff: regression invisible to seed, caught by grown set
check("regression invisible to seed set", seed_ok is True)
check("regression CAUGHT by grown set", grown_ok is False and len(grown_cf) > 0)

# 8. healthy retriever passes BOTH sets (no false alarms)
h_ok, _, _, _ = gate(GOLDEN, retrieve, CANARIES)
check("no false alarm on healthy retriever", h_ok is True)

print()
print("ALL PASS" if all(c for _, c in checks) else "SOME FAILED")
assert all(c for _, c in checks), "verification failed"

## 11 · Recap & what's next

**You built a data flywheel.** Raw production feedback → **triage** (four failure buckets) →
**curation gate** (verifiable, non-adversarial, non-duplicate, human-approved) → **promotion**
into golden + **canary** rows → a stronger L92 gate that caught a regression the seed set
couldn't see. The system now can't silently re-break a failure a real user already reported.

**The one idea to keep:** *feedback is for discovering failures, a curated golden set is for
measuring quality — never confuse the two.* Everything promotable must be verifiable; a
stranger's downvote is a hint, not a test case.

**Homework**
1. Add a **severity** score to feedback (e.g. by cluster size) and promote only high-severity
   clusters to `CANARIES`, the rest to ordinary golden rows.
2. Replace the rule-based `triage()` with a real LLM classifier via the `triage_llm` hook and
   compare bucket assignments — where do they disagree?
3. Wire this into L90's `rag-service`: log real `/ask` turns + a 👍/👎 endpoint, run the
   flywheel weekly, and open a PR that appends promoted rows to `eval/golden.jsonl`.

**Next lesson — L94: Red-teaming & safety evals.** We stop trusting *input*: adversarial
users and **prompt injection hidden inside retrieved documents**. We'll build a red-team suite
(jailbreaks, injection payloads planted in the corpus) and turn successful attacks into — you
guessed it — canary rows, exactly like today.